In [1]:
import pdfplumber
import re
import os

dieter_book = "paper4.pdf"  # George E. Dieter - Mechanical Metallurgy

def clean(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'\[\d+[\d,\-]*\]', '', text)
    text = re.sub(r'\(\w+ [&\w]+ \d{4}\)', '', text)
    text = re.sub(r'fig\.?\s*\d+\w*', '', text)
    text = re.sub(r'table\.?\s*\d+\w*', '', text)
    text = re.sub(r'equation\.?\s*\d+\w*', '', text)
    text = re.sub(r'\b\d{4}\b', '', text)
    text = re.sub(r'[^a-z\s.]', '', text)
    text = re.sub(r'\b\w{1,2}\b', '', text)
    text = re.sub(r'-\s+', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def extract_text(pdf_path, skip_pages=0):
    paper_text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages):
            if i < skip_pages:
                continue
            extracted = page.extract_text()
            if extracted:
                paper_text += extracted + " "
    return clean(paper_text)

all_text = ""

if not os.path.exists(dieter_book):
    print(f"Skipping {dieter_book} — file not found")
else:
    print(f" Reading Dieter book (skipping front matter)...")
    dieter_text = extract_text(dieter_book, skip_pages=30)  # skip front matter
    all_text += dieter_text + " "
    print(f" Words from Dieter book: {len(dieter_text.split())}")

# ── Final ──────────────────────────────────────────────────────
all_text = all_text.strip()
text = all_text

print(f"\n📊 Total words: {len(text.split())}")
print(f"📝 Sample: {text[:200]}")

 Reading Dieter book (skipping front matter)...
 Words from Dieter book: 138359

📊 Total words: 138359
📝 Sample: sec. introduction rials which behave elastically will have linear stressstrain relationship. rubber example material with nonlinear stressstrain relation ship that still satisfies the definition elast


In [7]:
len(text)

947386

In [8]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer

In [80]:
tokenizer = Tokenizer()

In [81]:
tokenizer.fit_on_texts([text]) #Assigning every word a token number.

In [82]:
len(tokenizer.word_index)

10154

In [84]:
max_seq = 15 #Sentences were going up to 100 words, which could cause overfitting, so set a max limit for sequence length.
input = []
for sentences in text.split('.'):
    tokenized_sentences = tokenizer.texts_to_sequences([sentences])[0]
    for i in range(1, len(tokenized_sentences)):
        seq = tokenized_sentences[max(0, i+1-max_seq) : i+1]  # sliding window
        
        input.append(seq)
#The above loop splits the complete text into sentences then, for each sentence, creates a row of data that can be used to train word predictor

In [86]:
max_len = max([len(x) for x in input])
max_len

15

In [87]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [88]:
padded = pad_sequences(input, maxlen = max_len, padding = 'pre') 
#We padded with zeroes to have same number of items in each sequence.

In [89]:
x = padded[:,:-1]
y = padded[:,-1]
#Extracted x and y

In [90]:
from tensorflow.keras.utils import to_categorical

In [91]:
yohe = to_categorical(y)
print(x.shape,
y.shape,
yohe.shape) #As this is a categorical problem, we one hot encoded 'y'.

(119009, 14) (119009,) (119009, 10155)


In [92]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM

In [97]:
model = Sequential()

model.add(Embedding(10155, 50)) #Sending a 50 dimensional vector for every word in X for each timestep to LSTM as X(t)
model.add(LSTM(150))# Sending a 150 dimension vector to the Dense Output Layer.
model.add(Dense(10155, activation = 'softmax')) # Applying softmax and giving us the probability for every word.

In [98]:
model.compile(loss = 'categorical_crossentropy', optimizer = 'adam', metrics = ["accuracy"])

In [102]:
history = model.fit(x, yohe, epochs=25, validation_split = 0.2)

Epoch 1/25
 123/2976 ━━━━━━━━━━━━━━━━━━━━ 1:24 29ms/step - accuracy: 0.7086 - loss: 1.4401

KeyboardInterrupt: 

In [38]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ (None, 14, 50)              │         250,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ (None, 14, 150)             │         120,600 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_1 (LSTM)                        │ (None, 150)                 │         180,600 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 5000)                │         755,000 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 3,918,602 (14.95 MB)

 Trainable params: 1,306,200 (4.98 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2,612,402 (9.97 MB)

In [101]:
import numpy as np

test = 'theoretical'

for i in range(20):
    tokenized = tokenizer.texts_to_sequences([test])[0]
    padded = pad_sequences([tokenized], maxlen = max_len, padding = 'pre')
    output = model.predict(padded)
    pos = np.argmax(output)
            
    for word,index in tokenizer.word_index.items():
        if index == pos:
            test = test + ' ' + word
               

print(test)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
theoretical treatments exhaustion creep for secondary creep period predominantly the cleavage time believed the effect temperature and the rate strain time


In [ ]:
print(len(text.split()))


In [ ]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))  # Should show your GPU